# Set-up

In [54]:
import os
import sys
import csv
import random
import torch
import pyfaidx
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm
from scipy.stats import spearmanr, pearsonr

import hashlib

# Generate random 10 character hash
def create_run_hash():
    random_str = ''.join(random.choices('abcdefghijklmnopqrstuvwxyz0123456789', k=10))
    run_hash = hashlib.md5(random_str.encode()).hexdigest()[:10]
    return run_hash

import seqpro as sp

from bpnetlite.bpnet import BPNet, CountWrapper, ProfileWrapper, ControlWrapper, _ProfileLogitScaling
from tangermeme.io import extract_loci
from tangermeme.predict import predict
from tangermeme.deep_lift_shap import deep_lift_shap, _nonlinear
from tangermeme.plot import plot_logo
from tangermeme.seqlet import recursive_seqlets
from tangermeme.annotate import annotate_seqlets
from tangermeme.io import read_meme

from matplotlib import pyplot as plt
import seaborn as sns; sns.set_style('white')

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# print tangermeme version
import tangermeme
print(tangermeme.__version__)

In [55]:
def log_softmax_profile(
    y_profile,
):
    # Log softmax the predicted profile
    z = y_profile.shape
    y_profile = y_profile.reshape(y_profile.shape[0], -1)
    y_profile = torch.nn.functional.log_softmax(y_profile, dim=-1)
    y_profile = y_profile.reshape(*z)
    return y_profile

In [56]:
os.chdir("/cellar/users/aklie/projects/igvf/ChromBetaNet")

In [57]:
path_out = "/cellar/users/aklie/projects/igvf/ChromBetaNet/scratch/2025_09_05"

# Load genome

In [58]:
chrom_sizes = "/cellar/users/aklie/data/ref/genomes/hg38/hg38.chrom.sizes"
chrom_sizes = pd.read_csv(chrom_sizes, header=None, sep='\t', names=['chrom', 'size'])
chrom_sizes_dict = chrom_sizes.set_index('chrom')['size'].to_dict()

In [59]:
genome_fasta = "/cellar/users/aklie/data/ref/genomes/hg38/hg38.fa"
genome = pyfaidx.Fasta(genome_fasta)

# Load peaks

In [60]:
# Load peaks
path_peaks = "scratch/2025_07_05/day15_ENP_MPRA_design/peaks_with_predictions.tsv"
peaks = pd.read_csv(path_peaks, sep="\t")
peaks = peaks[peaks["log_counts_pred"] >= 0]
peaks.head()

In [61]:
# Set up figure
fig, ax = plt.subplots(1, 1, figsize=(4.25, 4.25))

# Replace KDE with hexbin
ax.hexbin(
    peaks["log_counts"],
    peaks["log_counts_pred"],
    gridsize=200,
    mincnt=1,
    alpha=1
)

# Pearson r and spearman r
pcc_val, pcc_pval = pearsonr(peaks["log_counts_pred"], peaks["log_counts"])
spearman_val, spearman_pval = spearmanr(peaks["log_counts_pred"], peaks["log_counts"])

# Clean up plot
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("Log(Observed Counts)")
ax.set_ylabel("Log(Predicted Counts)")

# Top left annotation with stats
ax.text(
    0.05, 0.95,
    f"PCC = {pcc_val:.2f}\nSpearman = {spearman_val:.2f}",
    fontsize=9,
    transform=ax.transAxes,
    ha='left', va='top',
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2)
)

# 
fig.tight_layout()

# Load motifs

In [62]:
path_motifs = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/models/motifs/meme/combined.meme"
motifs = read_meme(path_motifs)
motif_names = np.array(list(motifs.keys()))

In [63]:
path_annot = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/models/motifs/tfs_initial.txt"
tmp = pd.read_csv(path_annot, header=None, sep='\t', names=["motif_id", "annotated_tf", "evalue"])
motif_id_mp = tmp.set_index("motif_id")["annotated_tf"]

# Load model

In [64]:
path_model = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/models/D15_ENP/fold_0/chrombpnet/0.5/models/chrombpnet_nobias.h5"
model = BPNet.from_chrombpnet(path_model).cuda().eval()
count_wrapper = CountWrapper(model).cuda().eval()

# Load sequences

In [65]:
# find the peaks that intersect with the isl1 enhancer coordinates
def intersects_with_peak(row, peaks):
    chrom = row['chrom']
    start = row['start']
    end = row['end']
    
    # Filter peaks for the same chromosome
    chrom_peaks = peaks[peaks["chrom"] == chrom]
    
    # Check if any peak overlaps with the enhancer region
    for _, peak in chrom_peaks.iterrows():
        peak_start = peak["start"]
        peak_end = peak["end"]
        if not (end < peak_start or start > peak_end):  # No overlap condition
            return True
    return False

# and add the peak coordinates to the dataframe
def get_peak_coordinates(row, peaks):
    chrom = row['chrom']
    start = row['start']
    end = row['end']
    
    # Filter peaks for the same chromosome
    chrom_peaks = peaks[peaks["chrom"] == chrom]
    
    # Check if any peak overlaps with the enhancer region
    for _, peak in chrom_peaks.iterrows():
        peak_start = peak["start"]
        peak_end = peak["end"]
        if not (end < peak_start or start > peak_end):  # No overlap condition
            return f"{chrom}:{peak_start}-{peak_end}"
    return None

In [66]:
# Read in the headers from each sequence to get the coordinates
path_fasta = "scratch/2025_07_05/day15_ENP_MPRA_design/ISL1_CRE.fa"
seqs = pyfaidx.Fasta(path_fasta)
seq_headers = list(seqs.keys())
isl1_enhancer_df = pd.DataFrame(seq_headers, columns=['header'])
isl1_enhancer_df['chrom'] = isl1_enhancer_df['header'].str.split(':').str[0]
isl1_enhancer_df['start'] = isl1_enhancer_df['header'].str.split(':').str[1].str.split('-').str[0].astype(int)
isl1_enhancer_df['end'] = isl1_enhancer_df['header'].str.split(':').str[1].str.split('-').str[1].astype(int)
isl1_enhancer_df['intersects_peak'] = isl1_enhancer_df.apply(intersects_with_peak, peaks=peaks, axis=1)
isl1_enhancer_df['peak_coordinates'] = isl1_enhancer_df.apply(get_peak_coordinates, peaks=peaks, axis=1)
isl1_enhancer_df["peak_chrom"] = isl1_enhancer_df["peak_coordinates"].str.split(":").str[0]
isl1_enhancer_df["peak_start"] = isl1_enhancer_df["peak_coordinates"].str.split(":").str[1].str.split("-").str[0].astype(int)
isl1_enhancer_df["peak_end"] = isl1_enhancer_df["peak_coordinates"].str.split(":").str[1].str.split("-").str[1].astype(int)
isl1_enhancer_df

In [67]:
# Annote peaks as Enhancer_1, Enhancer_2, etc.
peaks_selected = peaks.merge(isl1_enhancer_df[["peak_chrom", "peak_start", "peak_end"]], 
                                       left_on=["chrom", "start", "end"], 
                                       right_on=["peak_chrom", "peak_start", "peak_end"], 
                                       how="left", suffixes=("", "_enhancer"))
peaks_selected['CRE_annotation'] = np.where(
    peaks_selected['peak_chrom'].notnull(),
    'Enhancer',
    np.nan
)
peaks_selected['CRE_annotation'] = peaks_selected.groupby('chrom').cumcount() + 1
peaks_selected['CRE_annotation'] = np.where(
    peaks_selected['peak_chrom'].notnull(),
    'CRE_' + peaks_selected['CRE_annotation'].astype(str),
    np.nan
)

# Drop the temporary columns used for annotation
peaks_selected = peaks_selected.drop(columns=["peak_chrom", "peak_start", "peak_end"])
peaks_selected["CRE_annotation"].value_counts()

In [68]:
# Get the deciles of the distribution
deciles = np.percentile(peaks_selected["log_counts_pred"].dropna(), [0, 5, 15, 25, 35, 45, 55, 65, 75, 85, 95, 100])
deciles

In [69]:
# Which decile bin does each CRE belong to?
peaks_selected['decile'] = pd.cut(peaks_selected['log_counts_pred'], bins=deciles, labels=False)
peaks_selected["decile"].value_counts()

In [70]:
peaks_selected[~peaks_selected["CRE_annotation"].isna()]

In [71]:
# Plot a beautiful histogram of the predictions 
with sns.plotting_context("notebook", font_scale=1.5):
    fig, ax = plt.subplots(figsize=(8, 4))

    # Plot forward
    sns.histplot(
        peaks_selected["log_counts_pred"],
        bins=50,
        kde=True,
        color='blue',
        label='Predicted Counts',
        ax=ax
    )

    # Plot each ISL1 CRE as a vertical line each with a different color
    CRE_colors = sns.color_palette("husl", peaks_selected["CRE_annotation"].nunique())
    for i, (CRE_name, group) in enumerate(peaks_selected.groupby("CRE_annotation")):
        if CRE_name.startswith("CRE_"):
            CRE_color = CRE_colors[i % len(CRE_colors)]
            #Grab chr:start-end for label
            locus = f"{group['chrom'].iloc[0]}:{group['start'].iloc[0]}-{group['end'].iloc[0]}"
            ax.axvline(
                x=group["log_counts_pred"].mean(),
                color=CRE_color,
                linestyle='--',
                linewidth=2,
                label=locus
            )

    # Move legend outside of the plot
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.xlabel('Log(Tn5 insertion)')
    plt.ylabel('Frequency')
    plt.xticks(rotation=45)

    # 
    plt.tight_layout()

In [72]:
# Plot a beautiful histogram of the predictions with deciles
with sns.plotting_context("notebook", font_scale=1.5):
    fig, ax = plt.subplots(figsize=(8, 4))

    # Plot forward
    sns.histplot(
        peaks_selected["log_counts_pred"],
        bins=50,
        kde=True,
        color='blue',
        label='Predicted Counts',
        ax=ax
    )

    # Plot each ISL1 CRE as a vertical line each with a different color
    CRE_colors = sns.color_palette("husl", peaks_selected["CRE_annotation"].nunique())
    for i, (CRE_name, group) in enumerate(peaks_selected.groupby("CRE_annotation")):
        if CRE_name.startswith("CRE_"):
            CRE_color = CRE_colors[i % len(CRE_colors)]
            #Grab chr:start-end for label
            locus = f"{group['chrom'].iloc[0]}:{group['start'].iloc[0]}-{group['end'].iloc[0]}"
            ax.axvline(
                x=group["log_counts_pred"].mean(),
                color=CRE_color,
                linestyle='--',
                linewidth=2,
                label=locus
            )

    # Add vertical lines for deciles
    for decile in deciles:
        ax.axvline(decile, color='red', linestyle='-', linewidth=1)

    # Move legend outside of the plot
    #ax.legend(loc='lower center', bbox_to_anchor=(1, 1))
    plt.xlabel('Log(Tn5 insertion)')
    plt.ylabel('Frequency')
    plt.xticks(rotation=45)

    # 
    plt.tight_layout()

# CRE design

In [73]:
MPRA_integrated_sequence_1 = "gggtctctctggttagaccagatctgagcctgggagctctctggctaactagggaacccactgcttaagcctcaataaagcttgccttgagtgcttcaagtagtgtgtgcccgtctgttgtgtgactctggtaactagagatccctcagacccttttagtcagtgtggaaaatctctagcagtggcgcccgaacagggacttgaaagcgaaagggaaaccagaggagctctctcgacgcaggactcggcttgctgaagcgcgcacggcaagaggcgaggggcggcgactggtgagtacgccaaaaattttgactagcggaggctagaaggagagagatgggtgcgagagcgtcagtattaagcgggggagaattagatcgcgatgggaaaaaattcggttaaggccagggggaaagaaaaaatataaattaaaacatatagtatgggcaagcagggagctagaacgattcgcagttaatcctggcctgttagaaacatcagaaggctgtagacaaatactgggacagctacaaccatcccttcagacaggatcagaagaacttagatcattatataatacagtagcaaccctctattgtgtgcatcaaaggatagagataaaagacaccaaggaagctttagacaagatagaggaagagcaaaacaaaagtaagaccaccgcacagcaagcggccggccgctgatcttcagacctggaggaggagatatgagggacaattggagaagtgaattatataaatataaagtagtaaaaattgaaccattaggagtagcacccaccaaggcaaagagaagagtggtgcagagagaaaaaagagcagtgggaataggagctttgttccttgggttcttgggagcagcaggaagcactatgggcgcagcgtcaatgacgctgacggtacaggccagacaattattgtctggtatagtgcagcagcagaacaatttgctgagggctattgaggcgcaacagcatctgttgcaactcacagtctggggcatcaagcagctccaggcaagaatcctggctgtggaaagatacctaaaggatcaacagctcctggggatttggggttgctctggaaaactcatttgcaccactgctgtgccttggaatgctagttggagtaataaatctctggaacagatttggaatcacacgacctggatggagtgggacagagaaattaacaattacacaagcttaatacactccttaattgaagaatcgcaaaaccagcaagaaaagaatgaacaagaattattggaattagataaatgggcaagtttgtggaattggtttaacataacaaattggctgtggtatataaaattattcataatgatagtaggaggcttggtaggtttaagaatagtttttgctgtactttctatagtgaatagagttaggcagggatattcaccattatcgtttcagacccacctcccaaccccgaggggacccgacaggcccgaaggaatagaagaagaaggtggagagagagacagagacagatccattcgattagtgaacggatcggcactgcgtgcgccaattctgcagacaaatggcagtattcatccacaattttaaaagaaaaggggggattggggggtacagtgcaggggaaagaatagtagacataatagcaacagacatacaaactaaagaattacaaaaacaaattacaaaaattcaaaattttcgggtttattacagggacagcagagatccagtttggttagtaccgggcccggtgctttgctctgagccagcccaccagtttggaatgactcctttttatgacttgaattttcaagtataaagtctagtgctaaatttaatttgaacaactgtatagtttttgctggttgggggaaggaaaaaaaatggtggcagtgtttttttcagaattagaagtgaaatgaaaacttgttgtgtgtgaggatttctaatgacatgtggtggttgcatactgagtgaagccggtgagcattctgccatgtcaccccctcgtgctcagtaatgtactttacagaaatcctaaactcaaaagattgatataaaccatgcttcttgtgtatatccggtctcttctctgggtagtctcactcagcctgcatttctgccagggcccgctctagaccTGCAGGAGGACCGGATCAACTNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNCATTGCGTGAACCGACACTAGAGGGTATATAATGGAAGCTCGACTTCCAGCTTGGCAATCCGGTACTGTGCAAAGTGAACACATCGCTAAGCGAAAGCTAAGNNNNNNNNNNNNNNNACCGGtcgccaccatggtgagcaagggcgaggagctgttcaccggggtggtgcccatcctggtcgagctggacggcgacgtaaacggccacaagttcagcgtgtccggcgagggcgagggcgatgccacctacggcaagctgaccctgaagttcatctgcaccaccggcaagctgcccgtgccctggcccaccctcgtgaccaccctgacctacggcgtgcagtgcttcagccgctaccccgaccacatgaagcagcacgacttcttcaagtccgccatgcccgaaggctacgtccaggagcgcaccatcttcttcaaggacgacggcaactacaagacccgcgccgaggtgaagttcgagggcgacaccctggtgaaccgcatcgagctgaagggcatcgacttcaaggaggacggcaacatcctggggcacaagctggagtacaactacaacagccacaacgtctatatcatggccgacaagcagaagaacggcatcaaggtgaacttcaagatccgccacaacatcgaggacggcagcgtgcagctcgccgaccactaccagcagaacacccccatcggcgacggccccgtgctgctgcccgacaaccactacctgagcacccagtccgccctgagcaaagaccccaacgagaagcgcgatcacatggtcctgctggagttcgtgaccgccgccgggatcactctcggcatggacgagctgtacaagtaggaattcgtcgagggacctaataacttcgtatagcatacattatacgaagttatacatgtttaagggttccggttccactaggtacaattcgatatcaagcttatcgataatcaacctctggattacaaaatttgtgaaagattgactggtattcttaactatgttgctccttttacgctatgtggatacgctgctttaatgcctttgtatcatgctattgcttcccgtatggctttcattttctcctccttgtataaatcctggttgctgtctctttatgaggagttgtggcccgttgtcaggcaacgtggcgtggtgtgcactgtgtttgctgacgcaacccccactggttggggcattgccaccacctgtcagctcctttccgggactttcgctttccccctccctattgccacggcggaactcatcgccgcctgccttgcccgctgctggacaggggctcggctgttgggcactgacaattccgtggtgttgtcggggaaatcatcgtcctttccttggctgctcgcctgtgttgccacctggattctgcgcgggacgtccttctgctacgtcccttcggccctcaatccagcggaccttccttcccgcggcctgctgccggctctgcggcctcttccgcgtcttcgccttcgccctcagacgagtcggatctccctttgggccgcctccccgcatcgataccgtcgacctcgatcgagacctagaaaaacatggagcaatcacaagtagcaatacagcagctaccaatgctgattgtgcctggctagaagcacaagaggaggaggaggtgggttttccagtcacacctcaggtaccaagcatggggtaaagtactgttctcatcacatcatatcaaggttatataccatcaatattgccacagatgttacttagccttttaatatttctctaatttagtgtatatgcaatgatagttctctgatttctgagattgagtttctcatgtgtaatgattatttagagtttctctttcatctgttcaaatttttgtctagttttattttttactgatttgtaagacttctttttataatctgcatattacaattctctttactggggtgttgcaaatattttctgtcattctatggcctgacttttcttaatggttttttaattttaaaaataagtcttaatattcatgcaatctaattaacaatcttttctttgtggttaggactttgagtcataagaaatttttctctacactgaagtcatgatggcatgcttctatattattttctaaaagatttaaagttttgccttctccatttagacttataattcactggaatttttttgtgtgtatggtatgacatatgggttcccttttattttttacatataaatatatttccctgtttttctaaaaaagaaaaagatcatcattttcccattgtaaaatgccatatttttttcataggtcacttacatatatcaatgggtctgtttctgagctctactctattttatcagcctcactgtctatccccacacatctcatgctttgctctaaatcttgatatttagtggaacattctttcccattttgttctacaagaatatttttgttattgtcttttgggcttctatatacattttagaatgaggttggcaagttctgtagatcttagccactttttaaaagaaaaggggggactggaagggctaattcactcccaacgaagacaagatatccttgatctgtggatctaccacacacaaggctacttccctgattggcagaactacacaccagggccagggatcagatatccactgacctttggatggtgctacaagctagtaccagttgagcaagagaaggtagaagaagccaatgaaggagagaacacccgcttgttacaccctgtgagcctgcatgggatggatgacccggagagagaagtattagagtggaggtttgacagccgcctagcatttcatcacatggcccgagagctgcatccggactgtactgggtctctctggttagaccagatctgagcctgggagctctctggctaactagggaacccactgcttaagcctcaataaagcttgccttgagtgcttcaagtagtgtgtgcccgtctgttgtgtgactctggtaactagagatccctcagacccttttagtcagtgtggaaaatctctagca"
len(MPRA_integrated_sequence_1), MPRA_integrated_sequence_1.count('N')

In [74]:
# Define model and ism params
seq_len = 2114
target_len = 1000
ism_window = 270
plot_window = 270

# Seq limits
seq_center = 2114 // 2
seq_start = seq_center - ism_window // 2
seq_end = seq_center + ism_window // 2

# Target limits
target_center = 1000 // 2
target_start = target_center - ism_window // 2
target_end = target_center + ism_window // 2

# Plot starts
plot_seq_center = 1057
plot_seq_start = seq_center - plot_window // 2
plot_seq_end = seq_center + plot_window // 2
plot_target_center = 500
plot_target_start = target_center - plot_window // 2
plot_target_end = target_center + plot_window // 2

print(f"seq_start: {seq_start}, seq_end: {seq_end}")
print(f"target_start: {target_start}, target_end: {target_end}")
print(f"plot_seq_start: {plot_seq_start}, plot_seq_end: {plot_seq_end}")
print(f"plot_target_start: {plot_target_start}, plot_target_end: {plot_target_end}")

In [133]:
# Candidate
CRE_idx = 2

# Create run hash
run_hash = create_run_hash()
print(f"Run hash: {run_hash}")

In [134]:
# Prep
design_res = {}
CRE_ = peaks_selected[peaks_selected["CRE_annotation"].notnull()].iloc[CRE_idx]
idx_ = peaks_selected[peaks_selected["CRE_annotation"].notnull()].index[CRE_idx]
chrom_ = CRE_["chrom"]
start_ = CRE_["start"]
end_ = CRE_["end"]
summit_ = CRE_["peak"]
decile_ = CRE_["decile"]
preds_ = peaks_selected["log_counts_pred"].values[idx_]
print(f"Processing CRE: {chrom_}:{start_}-{end_}: {preds_}")

plot_coordinate_start = start_ + summit_ - plot_window // 2
plot_coordinate_end = plot_coordinate_start + plot_window
print(f"Plotting coordinates: {chrom_}:{plot_coordinate_start}-{plot_coordinate_end}")

# Params
dtype = torch.float64
k = 5
temp = 0.75
mode = "target"
delta = 0.1
max_iter = 15
n_runs = 100

In [135]:
# Create directory to save
dir_name = f"ISL1_{chrom_}:{start_}-{end_}_ISM_design_{run_hash}"
os.makedirs(os.path.join(path_out, dir_name), exist_ok=True)

## Reference prediction

In [136]:
# Verify prediction
ref_seq_start = start_ + summit_ - seq_len // 2
ref_seq_end = ref_seq_start + seq_len
design_res["reference_locus"] = f"{chrom_}:{ref_seq_start}-{ref_seq_end}"
ref_seq = genome[chrom_][ref_seq_start:ref_seq_end].seq.upper()
design_res["reference_sequence"] = ref_seq
X_ref = torch.tensor(sp.ohe([ref_seq], alphabet=sp.DNA).transpose(0, 2, 1), dtype=torch.uint8)
y_ref = predict(
    model=count_wrapper.type(torch.float32),
    X=X_ref,
    verbose=True,   
)
design_res["reference_prediction"] = y_ref.item()
design_res

## ISM design

In [137]:
# imports
import numpy as np
from tangermeme.predict import predict
from tangermeme.saturation_mutagenesis import saturation_mutagenesis


def greedy_ism(
    model,
    X,
    seq_len=2114,
    ism_window=500,
    k=1,
    temp=1.0,
    max_iter=10,
    verbose=False,
    mode="maximize",
    y_bar=None,
    delta=None,
    random_state=None
):
    """
    Perform greedy ISM on a sequence.

    Parameters
    ----------
    model : BPNet
        The model to use for prediction.
    X : torch.Tensor
        The input sequence as a one-hot encoded tensor.
    seq_len : int
        The length of the input sequence.
    ism_window : int
        The window size for ISM.
    k : int
        The number of top positions to select.
    max_iter : int
        The maximum number of iterations to perform.
    verbose : bool
        Whether to print progress messages.
    mode : str
        Whether to 'maximize' or 'minimize' the model score. Default is 'maximize'.

    Returns
    -------
    X_ : torch.Tensor
        The modified sequence after greedy ISM.
    """

    # Set up random number generator if random_state is not None
    if random_state is not None:
        if isinstance(random_state, int):
            rng = np.random.default_rng(random_state)
        elif isinstance(random_state, np.random.Generator):
            rng = random_state
    else:
        rng = np.random.default_rng()

    # Define seq limits
    seq_center = seq_len // 2
    seq_start = seq_center - ism_window // 2
    seq_end = seq_center + ism_window // 2

    # Initial copy of the tensor
    X_ = X.clone()

    # Get the original prediction
    y0 = predict(model, X_).cpu().numpy()[0]
    if verbose:
        print(f"Iteration 0/{max_iter} - y_hat: {y0[0]:.3f}")

    # Run ISM across rounds
    for i in range(max_iter):
        if verbose:
            print(f"Iteration {i+1}/{max_iter}", end="")

        # Run ISM
        y0, y_hat = saturation_mutagenesis(
            model,
            X_,
            start=seq_start,
            end=seq_end,
            raw_outputs=True,
            verbose=False,
        )

        # Get the updates
        if mode in ["maximize", "minimize"]:
            attr = y_hat[:, :, :, 0] - y0[:, None, None, 0]
        elif mode == "target":
            if y_bar is None:
                raise ValueError("y_bar must be specified when mode is 'target'")
            attr = -torch.abs(y_hat[:, :, :, 0] - y_bar)
        attr = attr.numpy()

        # Take the average across the batch
        if X.shape[0] == 2:
            attr_mean = np.mean(attr, axis=0)
        else:
            attr_mean = attr.squeeze(0)

        # Flip attribution if minimizing
        if mode == "minimize":
            attr_mean = -attr_mean

        # Get the top k positions
        idx = np.argsort(attr_mean.ravel())[:-k-1:-1]
        
        # if k==1, select the first
        if k == 1:
            indices = np.column_stack(np.unravel_index(idx, attr_mean.shape))[0]

        # if k>1, randomly select one of the indices, using temperature
        elif k > 1:
            idxs = np.column_stack(np.unravel_index(idx, attr_mean.shape))
            predictions = attr_mean[idxs[:, 0], idxs[:, 1]]
            probs = np.exp(predictions / temp)
            probs /= probs.sum()
            indices = rng.choice(np.column_stack(np.unravel_index(idx, attr_mean.shape)), size=1, p=probs)[0]

        # if k<1, raise an error
        else:
            raise ValueError("k must be greater than or equal to 1")

        # if verbose print out the new y_hat value
        if verbose:
            y_hat_np = y_hat[:, :, :, 0].cpu().numpy()
            print(f" - y_hat: {y_hat_np[0, indices[0], indices[1]]:.3f}")

        # Get the edit channel and position
        edit_channel = indices[0]
        edit_position = seq_start + indices[1]

        # zero out the position across the channels
        X_[0, :, edit_position] = 0
        X_[0, edit_channel, edit_position] = 1

        if mode == "target":
            if delta is not None:
                # Check to see if we are in delta of target, break if so
                curr_delta = np.abs(attr_mean[indices[0], indices[1]])
                if curr_delta < delta:
                    print(f"Breaking early at iteration {i+1}/{max_iter} - curr_delta: {curr_delta:.3f}")
                    break

    return X_

In [138]:
# Get a list of all deciles above _decile_
deciles_above = deciles[int(decile_)+1:]
print(f"Deciles above {decile_}: {deciles_above}")

y_bars = [torch.zeros(1, dtype=torch.float32) + d for d in deciles_above]
y_bars

In [139]:
X_hat = []
for y_bar in tqdm(y_bars, desc="Running ISM for each decile"):
    print(f"Running ISM for decile target: {y_bar.item()}")
    X_hat_ = []
    for i in tqdm(range(n_runs)):
        X_hat_.append(greedy_ism(
            count_wrapper,
            X_ref,
            seq_len=seq_len,
            ism_window=ism_window,
            k=k,
            temp=temp,
            mode="target",
            y_bar=y_bar,
            delta=delta,
            max_iter=max_iter,
            verbose=False
        ))
    X_hat.append(X_hat_)
X_hat = [torch.stack(x_hat_) for x_hat_ in X_hat]
X_hat = torch.stack(X_hat)
X_hat.shape

In [140]:
# Create identifiers for each of X_hat based on ISL1_{chrom_}:{start_}-{end_}_ISM_run_{run}_decile_{decile}
identifiers = []
for decile in range(X_hat.shape[0]):
    for run in range(X_hat.shape[1]):
        for t, seq in enumerate(range(X_hat.shape[2])):
            identifiers.append(f"ISL1_{chrom_}:{start_}-{end_}_ISM_run-{run+1}_target-{round(y_bars[decile].item(), 2)}_{t+1}")

In [141]:
# Stack the everything but the last two dims
X_optim = X_hat.view(-1, X_hat.shape[-2], X_hat.shape[-1])
X_optim.shape

In [142]:
y_hat = []
for j, y_bar in enumerate(y_bars):
    y0 = y_ref.numpy(force=True)[0, 0]
    batch_size = n_runs
    X_hat = X_optim[j*batch_size:(j+1)*batch_size]
    ax = plt.subplot(111)
    y_hat_ = predict(
        model=count_wrapper.type(torch.float32),
        X=X_hat,
        verbose=True,
    )
    plt.hist(y_hat_, facecolor='0.5', edgecolor='0.5')
    y_hat.append(y_hat_)
    plt.plot([y0, y0], [0, 5], label="Baseline prediction", c='c')
    plt.plot([y_bar, y_bar], [0, 5], label="Design target", c='m', linestyle=(0, (2, 2)))
    plt.title(f"ISM for {chrom_}:{start_}-{end_} with target {round(y_bar.item(), 2)}")
    plt.legend(loc=(1.01, 0.5))
    plt.xlabel("Predicted Log Counts")
    plt.ylabel("# Sequences")
    sns.despine(left=True)
    plt.grid(False)
    plt.setp(ax.spines.values(), linewidth=2, color='0.5')
    plt.tight_layout()
    plt.savefig(os.path.join(path_out, dir_name, f"ISL1_{chrom_}:{start_}-{end_}_ISM_design_target_{round(y_bar.item(), 2)}_{run_hash}.pdf"), dpi=300)
    plt.show()
y_hat = torch.stack(y_hat).squeeze()
y_hat = y_hat.view(-1)

In [143]:
optim_seqs = sp.decode_ohe(X_optim.detach().cpu().numpy().astype(np.uint8), alphabet=sp.DNA, ohe_axis=1)
optim_seqs = [b''.join(optim_seq).decode('utf-8') for optim_seq in optim_seqs]
optim_seqs[:5]

## Calculate attributions and call seqlets

In [144]:
# Get attribution scores for the original sequence
X_ref_attr_ = deep_lift_shap(
    count_wrapper.type(dtype), 
    X=X_ref.type(dtype),
    n_shuffles=20,
    batch_size=16,
    verbose=True,
    warning_threshold=1e-4
)
X_ref_attr_.shape

In [145]:
# Get attribution scores for the original sequence
X_optim_attr_ = deep_lift_shap(
    count_wrapper.type(dtype), 
    X=X_optim.type(dtype),
    n_shuffles=20,
    batch_size=16,
    verbose=True,
    warning_threshold=1e-4
)
X_optim_attr_.shape

In [146]:
# Save attribution scores to npz
np.savez_compressed(os.path.join(path_out, dir_name, f"ISL1_{chrom_}:{start_}-{end_}_ISM_X_ref_attr_{run_hash}.npz"), X_ref_attr=X_ref_attr_.numpy(force=True))
np.savez_compressed(os.path.join(path_out, dir_name, f"ISL1_{chrom_}:{start_}-{end_}_ISM_X_optim_attr_{run_hash}.npz"), X_optim_attr=X_optim_attr_.numpy(force=True))

In [147]:
# Call and annotate seqlets
X_attr_ = torch.cat([X_ref_attr_, X_optim_attr_], dim=0)
total_attr_mp = pd.Series(index=["ISL1_chr5:51382751-51383709_ref_sequence"] + identifiers, data=torch.abs(X_attr_).sum(dim=(-2, -1)).numpy())
X_ = torch.cat([X_ref.cpu(), X_optim.cpu()], dim=0)
seqlets = recursive_seqlets(X_attr_[:, :, seq_start:seq_end].sum(dim=1).detach(), threshold=0.05, additional_flanks=5)
motif_idxs = annotate_seqlets(X_[:, :, seq_start:seq_end].detach().cpu(), seqlets, path_motifs)[0][:, 0]
seqlets['start'] += seq_start
seqlets['end'] += seq_start
motifs_ = motif_id_mp.loc[motif_names[motif_idxs]]
seqlets["annotation"] = motifs_.values
seqlets["id"] = [(["ISL1_chr5:51382751-51383709_ref_sequence"] + identifiers)[idx] for idx in seqlets["example_idx"]]

# calcualte the proportion of total attribution for each seqlet
seqlets["attribution_proportion"] = seqlets["attribution"].abs() / seqlets["id"].map(total_attr_mp)
seqlets.head()

In [148]:
# Save
seqlets[["id", "start", "end", "attribution", "p-value", "annotation"]].sort_values(["id", "start"]).to_csv(
    os.path.join(path_out, dir_name, f"ISL1_{chrom_}:{start_}-{end_}_ISM_seqlets_{run_hash}.tsv"), 
    sep="\t", 
    index=False
)

In [149]:
def overlap_len(a_start, a_end, b_start, b_end):
    return max(0, min(a_end, b_end) - max(a_start, b_start))

def greedy_match(curr_design_df, baseline_df):
    cand = (
        curr_design_df.reset_index().merge(
            baseline_df.reset_index(),
            on="annotation",
            suffixes=("_d","_b")
        )
    )
    if cand.empty:
        return [], set(), set()

    cand["overlap"] = cand.apply(
        lambda r: overlap_len(r["start_d"], r["end_d"], r["start_b"], r["end_b"]),
        axis=1
    )
    cand = cand[cand["overlap"] > 0]
    if cand.empty:
        return [], set(), set()

    cand["start_gap"] = (cand["start_d"] - cand["start_b"]).abs()
    cand = cand.sort_values(["overlap","start_gap"], ascending=[False,True])

    used_d, used_b, matches = set(), set(), []
    for _, r in cand.iterrows():
        di, bi = r["index_d"], r["index_b"]
        if di not in used_d and bi not in used_b:
            used_d.add(di); used_b.add(bi)
            matches.append((di, bi))
    return matches, used_d, used_b

In [150]:
ref_seqlets = seqlets[seqlets["id"].str.contains("_ref_sequence")].copy()
designs_seqlets  = seqlets[~seqlets["id"].str.contains("_ref_sequence")].copy()
ref_seqlets.shape, designs_seqlets["id"].nunique()

In [151]:
# Build delta-seqlets once (designs_seqlets vs ref_seqlets)
rows = []

for design_id, curr_design_df in designs_seqlets.groupby("id", sort=False):
    # work on copies to avoid SettingWithCopy surprises
    d = curr_design_df[["start","end","attribution","attribution_proportion","p-value","annotation"]].copy()
    b = ref_seqlets[      ["start","end","attribution","attribution_proportion","p-value","annotation"]].copy()

    # ensure numeric
    d["start"] = d["start"].astype(int);  d["end"] = d["end"].astype(int);  d["attribution"] = d["attribution"].astype(float)
    b["start"] = b["start"].astype(int);  b["end"] = b["end"].astype(int);  b["attribution"] = b["attribution"].astype(float)

    matches, used_d, used_b = greedy_match(d, b)

    # matched → increase/decrease
    for di, bi in matches:
        drow, brow = d.loc[di], b.loc[bi]
        delta_attr = float(drow["attribution"]) - float(brow["attribution"])
        change_type = "increase" if delta_attr > 0 else ("decrease" if delta_attr < 0 else "increase")
        rows.append({
            "id": design_id,
            "start": int(drow["start"]),
            "end": int(drow["end"]),
            "attribution": float(drow["attribution"]),
            "p-value": float(drow["p-value"]),
            "attribution_proportion": float(drow["attribution_proportion"]),
            "delta_attribution": float(delta_attr),
            "type": change_type,
            "annotation": drow["annotation"],
        })

    # additions → present in design only
    if len(d) > 0 and len(used_d) > 0:
        d_unmatched = d.drop(index=list(used_d))
    else:
        d_unmatched = d if len(d) else d.iloc[0:0]
    for _, drow in d_unmatched.iterrows():
        rows.append({
            "id": design_id,
            "start": int(drow["start"]),
            "end": int(drow["end"]),
            "attribution": float(drow["attribution"]),
            "p-value": float(drow["p-value"]),
            "attribution_proportion": float(drow["attribution_proportion"]),
            "delta_attribution": float(drow["attribution"]),
            "type": "addition",
            "annotation": drow["annotation"],
        })

    # removals → present in ref only
    if len(b) > 0 and len(used_b) > 0:
        b_unmatched = b.drop(index=list(used_b))
    else:
        b_unmatched = b if len(b) else b.iloc[0:0]
    for _, brow in b_unmatched.iterrows():
        rows.append({
            "id": design_id,
            "start": int(brow["start"]),
            "end": int(brow["end"]),
            "attribution": float("nan"),
            "p-value": float("nan"),
            "attribution_proportion": float("nan"),
            "delta_attribution": -float(brow["attribution"]),
            "type": "removal",
            "annotation": brow["annotation"],
        })

delta_seqlets_all = pd.DataFrame(rows)
delta_seqlets_all["type"] = pd.Categorical(
    delta_seqlets_all["type"],
    categories=["increase","decrease","addition","removal"],
    ordered=True,
)
delta_seqlets_all.head()

In [152]:
# Save
delta_seqlets_all.sort_values(["id", "start"]).to_csv(
    os.path.join(path_out, dir_name, f"ISL1_{chrom_}:{start_}-{end_}_ISM_delta_seqlets_{run_hash}.tsv"), 
    sep="\t", 
    index=False
)

## Quantify edits

In [153]:
# Want the following long data frame to quantify the differences
# Column name	Example	Description
# id	ISL1_chr5:51382751-51383709_ISM_run-1_target-10.88	Designed sequence identifier of form ISL1_{peak_chromosome}:{peak_start_coordinate}-{peak_end_coordinate}_{method}_run-{run}_target-{target}
# pos	1069	Position from start of input sequence where the edit occurs
# ref	C	Nucleotide in the ref_sequence corresponding to this id
# alt	G	Nucleotide in the design_sequence corresponding to this id

# Diffs is a list of lists of tuples and we need to match up each list with id
diffs = []
for j in range(len(optim_seqs)):
    diffs_ = [(i, ref_seq[i], optim_seqs[j][i]) for i in range(len(ref_seq)) if ref_seq[i] != optim_seqs[j][i]]
    diffs.append(diffs_)

# Make df
edit_df = []
for i, diff_list in enumerate(diffs):
    id_ = identifiers[i]
    for pos, ref, alt in diff_list:
        edit_df.append({
            'id': id_,
            'relative_pos': pos,
            'ref': ref,
            'alt': alt
        })
edit_df = pd.DataFrame(edit_df)
edit_df["chrom"] = chrom_
edit_df["pos"] = edit_df["relative_pos"] + ref_seq_start + 1
edit_df["variant_id"] = edit_df["chrom"] + ":" + edit_df["pos"].astype(str) + ":" + edit_df["ref"] + ":" + edit_df["alt"]
edit_df["id"].value_counts()

In [154]:
# Save edit_df
edit_df[["id", "chrom", "pos", "ref", "alt", "variant_id", "relative_pos"]].to_csv(os.path.join(path_out, dir_name, f"ISL1_{chrom_}:{start_}-{end_}_ISM_edits_{run_hash}.tsv"), sep="\t", index=False)

## Predict on MPRA integrated sequence context

In [155]:
# Get insert sequences by taking the middle 270 of each optim seq
insert_len = 270
barcode_len = 15
insert_seqs = [optim_seq[seq_center - insert_len // 2: seq_center + insert_len // 2] for optim_seq in optim_seqs]
results = []

# For each insert sequence, create the full MPRA sequence by inserting into MPRA_integrated_sequence_1, then chop at the middle 2114 and predict
for i, insert_seq in enumerate(insert_seqs):

    # Get Id
    curr_id = identifiers[i]

    # Get a random 15bp barcode sequence
    barcode_seq = ''.join(random.choices('ACGT', k=barcode_len))

    # Find the start and end of the 270 'N's in a row
    insert_start = MPRA_integrated_sequence_1.find('N' * insert_len)
    insert_end = insert_start + insert_len
    MPRA_seq = MPRA_integrated_sequence_1[:insert_start] + insert_seq + MPRA_integrated_sequence_1[insert_end:]

    # Find the start and end of exactly 15 'N's in a row and no more than 15 N's
    barcode_start = MPRA_seq.find('N' * barcode_len)
    barcode_end = barcode_start + barcode_len
    MPRA_seq = MPRA_seq[:barcode_start] + barcode_seq + MPRA_seq[barcode_end:]

    # Create seq by inserting insert_seq and barcode_seq into MPRA_integrated_sequence_1 at proper positions
    MPRA_seq = MPRA_seq.upper()

    # Take the middle seq_len of MPRA_seq
    insert_center = insert_start + insert_len // 2
    mpra_seq_start = insert_center - seq_len // 2
    mpra_seq_end = insert_center + seq_len // 2
    mpra_seq = MPRA_seq[mpra_seq_start:mpra_seq_end]

    # One-hot encode and predict
    X_mpra = torch.tensor(sp.ohe([mpra_seq], alphabet=sp.DNA).transpose(0, 2, 1), dtype=torch.uint8)
    y_mpra = predict(
        model=count_wrapper.type(torch.float32),
        X=X_mpra,
        verbose=False
    )

    # Save results
    results.append({
        "id": curr_id,
        "mpra_sequence": mpra_seq,
        "mpra_prediction": float(y_mpra.item())
    })

# Collect into dataframe
mpra_df = pd.DataFrame(results)
mpra_df.head()

## Save overall results

In [156]:
# Create a dataframe with the following specs
# Column name	Example	Description
# id	ISL1_chr5:51382751-51383709_ISM_run-1_target-10.88	Designed sequence identifier of form ISL1_{peak_chromosome}:{peak_start_coordinate}-{peak_end_coordinate}_{method}_run-{run}_target-{target}
# reference_locus	chr5:51382751-51383709	BED coordinates for the peak that was used to generate the designed sequence
# reference_sequence	TCAACC…	Reference genome sequence centered on the summit of reference_locus peak. Length corresponds to model input length
# reference_prediction	8.27	Prediction from model on reference_sequence
# baseline_sequence	ACCAAC…	MPRA integrated sequence with genomic insert
# baseline_prediction	8.39	Prediction from model on baseline_sequence
# design_method	ISM	Method used to design sequence
# design_sequence	ACGAA…	MPRA integrated sequence with designed insert
# design_target_prediction	10.88	Target prediction for design process
# design_prediction	10.15	Prediction from model on design_sequence
design_df = pd.DataFrame({
    "id": identifiers,
    "reference_locus": design_res["reference_locus"],
    "reference_sequence": design_res["reference_sequence"],
    "reference_prediction": design_res["reference_prediction"],
    "design_method": "ISM",
    "design_target": np.concatenate([np.repeat(y_bar.item(), n_runs) for y_bar in y_bars]),
    "design_sequence": optim_seqs,
    "design_prediction": y_hat.numpy(force=True),
})
design_df.head()

In [157]:
# merge designed_df with edited_df on id
designed_merged_df = design_df.merge(mpra_df, on="id")
designed_merged_df.head()

In [158]:
# Overwrite design_sequence and design_prediction with edited versions
header = {
    "method": "ISM",
    "n_runs": n_runs,
    "k": k,
    "temp": temp,
    "mode": mode,
    "delta": delta,
    "max_iter": max_iter,
    "n_runs": n_runs,
    "run_hash": run_hash
}
with open(os.path.join(path_out, dir_name, f"ISL1_{chrom_}:{start_}-{end_}_ISM_design_{run_hash}.tsv"), 'w', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    for key, value in header.items():
        writer.writerow([f"# {key}: {value}"])
    designed_merged_df.to_csv(f, sep="\t", index=False, float_format='%.4f')

In [159]:
# Save X_optim to npz
np.savez_compressed(os.path.join(path_out, dir_name, f"ISL1_{chrom_}:{start_}-{end_}_ISM_X_optim_{run_hash}.npz"), X_optim=X_optim.numpy(force=True))

# Plot them all

In [160]:
plots_dir = os.path.join(path_out, dir_name, "plots")
os.makedirs(plots_dir, exist_ok=True)

In [161]:
for seq_num in range(len(optim_seqs)):
    print(f"Processing sequence {seq_num+1}/{len(optim_seqs)}")

    # --- Get sequences ---
    original_seq = sp.decode_ohe(
        X_ref.cpu().numpy().astype(np.uint8), 
        alphabet=sp.DNA, ohe_axis=1
    )
    original_seq = b''.join(original_seq[0]).decode('utf-8')

    mutated_seq = sp.decode_ohe(
        np.expand_dims(X_optim[seq_num].detach().cpu().numpy().astype(np.uint8), axis=0),
        alphabet=sp.DNA, ohe_axis=1
    )
    mutated_seq = b''.join(mutated_seq[0]).decode('utf-8')

    diffs = [
        (i, original_seq[i], mutated_seq[i]) 
        for i in range(len(original_seq)) if original_seq[i] != mutated_seq[i]
    ]

    # --- Get predictions ---
    original_profile, original_counts = predict(model, X_ref)
    original_counts = np.squeeze(np.exp(original_counts.cpu().detach().numpy()))
    original_profile = np.squeeze(torch.exp(log_softmax_profile(original_profile)).cpu().detach().numpy())

    mutated_profile, mutated_counts = predict(model, X_optim[seq_num:seq_num+1])
    mutated_counts = np.squeeze(np.exp(mutated_counts.cpu().detach().numpy()))
    mutated_profile = np.squeeze(torch.exp(log_softmax_profile(mutated_profile)).cpu().detach().numpy())

    original_pred = original_counts * original_profile
    mutated_pred = mutated_counts * mutated_profile

    # --- Get seqlets ---
    plot_design_seqlets = seqlets[seqlets["id"] == identifiers[seq_num]] \
        .rename({"annotation": "motif_name", "attribution": "score"}, axis=1)
    plot_ref_seqlets = seqlets[seqlets["id"] == "ISL1_chr5:51382751-51383709_ref_sequence"] \
        .rename({"annotation": "motif_name", "attribution": "score"}, axis=1)

    plot_design_seqlets = plot_design_seqlets[["motif_name", "start", "end", "score"]]
    plot_ref_seqlets = plot_ref_seqlets[["motif_name", "start", "end", "score"]]

    # --- Plot ---
    with sns.plotting_context("paper", font_scale=1.5):
        fig, axes = plt.subplots(
            nrows=3, ncols=1, figsize=(18, 8), sharex=True,
            gridspec_kw={'height_ratios': [2, 1, 1]}
        )

        # Predictions
        axes[0].plot(
            range(0, plot_window),
            original_pred[plot_target_start:plot_target_end],
            color='r', linewidth=1, label=f"Original ({round(float(np.log(original_counts)), 2)})"
        )
        axes[0].plot(
            range(0, plot_window),
            mutated_pred[plot_target_start:plot_target_end],
            color='b', linewidth=1, label=f"Mutated ({round(float(np.log(mutated_counts)), 2)})"
        )
        axes[0].set_ylabel("Predicted Accessibility")
        axes[0].legend(fontsize=12, loc="upper left")
        axes[0].set_title(f"{chrom_}:{plot_coordinate_start}-{plot_coordinate_end}")

        # Attributions baseline
        plot_logo(
            X_ref_attr_[0], ax=axes[1],
            start=plot_seq_start, end=plot_seq_end,
            annotations=plot_ref_seqlets
        )
        axes[1].set_ylabel("Attributions")
        axes[1].set_ylim(-0.05, 0.15)

        # Attributions mutated
        plot_logo(
            X_optim_attr_[seq_num].detach().cpu(), ax=axes[2],
            start=plot_seq_start, end=plot_seq_end,
            annotations=plot_design_seqlets
        )
        axes[2].set_ylabel("Attributions")
        axes[2].set_ylim(-0.05, 0.15)

        # Mark edits
        for diff in diffs:
            pos = diff[0]
            plot_pos = pos - plot_seq_start
            axes[1].axvline(plot_pos, color='r', linestyle='--', linewidth=1)
            axes[2].axvline(plot_pos, color='r', linestyle='--', linewidth=1)

        plt.tight_layout()

        # Save figure
        out_path = os.path.join(plots_dir, f"{identifiers[seq_num]}_plot.pdf")
        plt.savefig(out_path, format="pdf")
        plt.close(fig)

        print(f"Saved plot → {out_path}")


# DONE!

---